In [12]:
import pandas as pd
import re


In [13]:
df = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/teleconsultations.csv", dtype={"mobile": "string"})

df.head()

,teleconsult_id,tele_source_patient_id,patient_name,gender,age,mobile,village,block,district,consult_date,...,consult_mode,duration_min,medicine_advised,test_advised,review_advised,review_due_days,distance_to_facility_km,preferred_language,connectivity_quality,consult_status
0,TC0000001,TEP0000001,Mohan Basha,M,47,9983609751,Chandur,Chandur,Nalgonda,2026-05-19,...,Audio,15,Yes,Yes,No,NaN,29.7,Telugu,Poor,Completed
1,TC0000002,TEP0000002,Mohan Laxman,M,44,9136944279,Bodajanampet,Balanagar,Mahabubnagar,2026-07-22,...,Assisted video,18,No,No,Yes,21.0,33.3,Telugu,Fair,Completed
2,TC0000003,TEP0000002,Mohan Laxman,M,47,<NA>,Bodajanampet,Balanagar,Mahabubnagar,2026-08-11,...,Video,20,No,No,Yes,7.0,27.6,Telugu,Good,Completed
3,TC0000004,TEP0000003,Mohan Narsimha,M,46,9484336710,Kondamallepally,Devarakonda,Nalgonda,2026-06-02,...,Video,16,Yes,Yes,Yes,30.0,25.7,Telugu,Fair,Completed
4,TC0000005,TEP0000004,Sai Krishna,M,19,9445488147,Macharam,Jadcherla,Mahabubnagar,2026-04-24,...,Video,19,Yes,Yes,Yes,14.0,17.2,Urdu,Fair,Completed


In [14]:
df["mobile_original"] = df["mobile"]


In [15]:
def clean_mobile(phone):
    if pd.isna(phone):
        return pd.NA

    phone = str(phone).strip()

    if phone == "":
        return pd.NA

    phone = re.sub(r"\.0$", "", phone)

    phone = re.sub(r"\D", "", phone)

    if phone.startswith("0091"):
        phone = phone[4:]

    elif phone.startswith("91") and len(phone) == 12:
        phone = phone[2:]

    if len(phone) != 10:
        return pd.NA

    if phone[0] not in "6789":
        return pd.NA

    return phone

In [16]:
df["mobile"] = df["mobile"].apply(clean_mobile)

In [17]:
df[
    ["mobile_original", "mobile"]
].head(100)

,mobile_original,mobile
0,9983609751,9983609751
1,9136944279,9136944279
2,<NA>,NaN
3,9484336710,9484336710
4,9445488147,9445488147
...,...,...
95,9600674893,9600674893
96,+91-95198-99121,9519899121
97,9519899121,9519899121
98,+91-98241-78781,9824178781


In [18]:

import pandas as pd
import re

base_path = "caseathon_data/Infinum_2026_Candidate_Dataset_Pack"

tele = pd.read_csv(f"{base_path}/teleconsultations.csv", dtype="string")
ncd = pd.read_csv(f"{base_path}/ncd_screening.csv", dtype="string")
rx = pd.read_csv(f"{base_path}/prescriptions.csv", dtype="string")
pharm = pd.read_csv(f"{base_path}/medicine_dispensing.csv", dtype="string")
visits = pd.read_csv(f"{base_path}/followup_visits.csv", dtype="string")
labs = pd.read_csv(f"{base_path}/lab_tests.csv", dtype="string")
outreach = pd.read_csv(f"{base_path}/outreach_actions.csv", dtype="string")
patients = pd.read_csv(f"{base_path}/patient_360_reference.csv", dtype="string")

In [19]:
datasets = {
    "tele": tele,
    "ncd": ncd,
    "rx": rx,
    "pharm": pharm,
    "visits": visits,
    "labs": labs,
    "outreach": outreach
}

for name, data in datasets.items():
    mobile_columns = [
        col for col in data.columns
        if "mobile" in col.lower() or "phone" in col.lower()
    ]

    for col in mobile_columns:
        data[f"{col}_original"] = data[col]
        data[col] = data[col].apply(clean_mobile)

        print(
            f"{name}.{col}: "
            f"{data[col].notna().sum()} valid, "
            f"{data[col].isna().sum()} missing/invalid"
        )

tele.mobile: 5078 valid, 438 missing/invalid
ncd.mobile: 4348 valid, 400 missing/invalid
rx.mobile: 4742 valid, 409 missing/invalid
pharm.mobile: 3455 valid, 301 missing/invalid
visits.mobile: 2392 valid, 183 missing/invalid
labs.mobile: 1857 valid, 148 missing/invalid
outreach.mobile: 5618 valid, 470 missing/invalid


In [20]:
for name, data in datasets.items():
    mobile_columns = [
        col for col in data.columns
        if "mobile" in col.lower()
        or "phone" in col.lower()
    ]

    print(f"\n{name}")

    for col in mobile_columns:
        print(
            f"{col}: "
            f"{data[col].notna().sum()} valid / "
            f"{len(data)} total"
        )


tele
mobile: 5078 valid / 5516 total
mobile_original: 5078 valid / 5516 total

ncd
mobile: 4348 valid / 4748 total
mobile_original: 4348 valid / 4748 total

rx
mobile: 4742 valid / 5151 total
mobile_original: 4742 valid / 5151 total

pharm
mobile: 3455 valid / 3756 total
mobile_original: 3455 valid / 3756 total

visits
mobile: 2392 valid / 2575 total
mobile_original: 2392 valid / 2575 total

labs
mobile: 1857 valid / 2005 total
mobile_original: 1857 valid / 2005 total

outreach
mobile: 5618 valid / 6088 total
mobile_original: 5618 valid / 6088 total


In [21]:
datasets = {
    "tele": tele,
    "ncd": ncd,
    "rx": rx,
    "pharm": pharm,
    "visits": visits,
    "labs": labs,
    "outreach": outreach,
    "patients": patients
}

for name, data in datasets.items():
    print("\n" + "=" * 80)
    print(f"{name.upper()} DATAFRAME")
    print("=" * 80)
    print("Shape:", data.shape)
    print("Columns:", list(data.columns))
    print()
    print(data.to_string())


TELE DATAFRAME
Shape: (5516, 24)
Columns: ['teleconsult_id', 'tele_source_patient_id', 'patient_name', 'gender', 'age', 'mobile', 'village', 'block', 'district', 'consult_date', 'facility_id', 'chief_complaint', 'diagnosis_group', 'consult_mode', 'duration_min', 'medicine_advised', 'test_advised', 'review_advised', 'review_due_days', 'distance_to_facility_km', 'preferred_language', 'connectivity_quality', 'consult_status', 'mobile_original']

     teleconsult_id tele_source_patient_id        patient_name gender age      mobile          village        block      district consult_date facility_id       chief_complaint        diagnosis_group    consult_mode duration_min medicine_advised test_advised review_advised review_due_days distance_to_facility_km preferred_language connectivity_quality consult_status  mobile_original
0         TC0000001             TEP0000001         Mohan Basha      M  47  9983609751          Chandur      Chandur      Nalgonda   2026-05-19      FAC015    Routine 

In [22]:
tele = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/teleconsultations.csv", dtype="string")

outcomes = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/episode_outcomes.csv", dtype="string")

patients = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/patient_360_reference.csv", dtype="string")

ncd = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/ncd_screening.csv", dtype="string")

rx = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/prescriptions.csv", dtype="string")

pharm = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/medicine_dispensing.csv", dtype="string")

labs = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/lab_tests.csv", dtype="string")

visits = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/followup_visits.csv", dtype="string")

outreach = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/outreach_actions.csv", dtype="string")

geo = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/geography_reference.csv", dtype="string")

stock = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/medicine_stock_status.csv", dtype="string")

facility = pd.read_csv("caseathon_data/Infinum_2026_Candidate_Dataset_Pack/facility_reference.csv", dtype="string")

In [23]:
print(patients.columns.tolist())
display(patients.head(10))

['patient_id', 'canonical_name', 'gender', 'date_of_birth', 'age_as_of_2026', 'masked_mobile', 'village_id', 'village', 'block', 'district', 'preferred_language', 'known_ncd_status', 'vulnerability_group', 'household_id']


,patient_id,canonical_name,gender,date_of_birth,age_as_of_2026,masked_mobile,village_id,village,block,district,preferred_language,known_ncd_status,vulnerability_group,household_id
0,P000001,Mohan Mallaiah,M,2008-05-18,18,XXXXXX5988,VIL0035,Indresham,Patancheru,Sangareddy,Telugu,Hypertension+Diabetes,General,HH00384
1,P000002,Shankar Rao,M,1997-07-03,29,XXXXXX5269,VIL0007,Kallepally,Balanagar,Mahabubnagar,Telugu,<NA>,General,HH01266
2,P000003,Mohan Basha,M,1979-08-22,47,XXXXXX9751,VIL0017,Chandur,Chandur,Nalgonda,Telugu,<NA>,Low digital access,HH00709
3,P000004,Mohan Laxman,M,1981-02-20,45,XXXXXX4279,VIL0008,Bodajanampet,Balanagar,Mahabubnagar,Telugu,Hypertension,General,HH01940
4,P000005,Mohan Narsimha,M,1980-12-18,46,XXXXXX6710,VIL0014,Kondamallepally,Devarakonda,Nalgonda,Telugu,Hypertension,Elderly living alone,HH02488
5,P000006,Sai Krishna,M,2008-04-07,18,XXXXXX8147,VIL0004,Macharam,Jadcherla,Mahabubnagar,Urdu,Hypertension,Seasonal migrant,HH01068
6,P000007,Anitha Narsimha,F,2008-12-28,18,XXXXXX2238,VIL0013,Devarakonda,Devarakonda,Nalgonda,Telugu,Hypertension+Diabetes,Elderly living alone,HH00499
7,P000008,Ramesh Begum,M,1960-01-22,66,XXXXXX0540,VIL0026,Kangti,Narayankhed,Sangareddy,Telugu,<NA>,General,HH02129
8,P000009,Suresh Reddy,M,1983-02-18,43,XXXXXX2019,VIL0017,Chandur,Chandur,Nalgonda,Telugu,Hypertension+Diabetes,Low digital access,HH02493
9,P000010,Bhavani Mudiraj,F,1965-08-25,61,XXXXXX1512,VIL0033,Patancheru,Patancheru,Sangareddy,Telugu,<NA>,General,HH00475


In [25]:
def normalize_text(value):
    if pd.isna(value):
        return ""

    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9\s]", "", value)
    value = re.sub(r"\s+", " ", value)

    return value


patients["name_norm"] = patients["canonical_name"].apply(normalize_text)
patients["village_norm"] = patients["village"].apply(normalize_text)
patients["block_norm"] = patients["block"].apply(normalize_text)
patients["district_norm"] = patients["district"].apply(normalize_text)

patients["mobile_last4"] = (
    patients["masked_mobile"]
    .astype("string")
    .str.extract(r"(\d{4})$", expand=False)
)

In [26]:
display(
    patients[
        [
            "patient_id",
            "canonical_name",
            "mobile_last4",
            "village",
            "block",
            "district"
        ]
    ].head(10)
)

,patient_id,canonical_name,mobile_last4,village,block,district
0,P000001,Mohan Mallaiah,5988,Indresham,Patancheru,Sangareddy
1,P000002,Shankar Rao,5269,Kallepally,Balanagar,Mahabubnagar
2,P000003,Mohan Basha,9751,Chandur,Chandur,Nalgonda
3,P000004,Mohan Laxman,4279,Bodajanampet,Balanagar,Mahabubnagar
4,P000005,Mohan Narsimha,6710,Kondamallepally,Devarakonda,Nalgonda
5,P000006,Sai Krishna,8147,Macharam,Jadcherla,Mahabubnagar
6,P000007,Anitha Narsimha,2238,Devarakonda,Devarakonda,Nalgonda
7,P000008,Ramesh Begum,0540,Kangti,Narayankhed,Sangareddy
8,P000009,Suresh Reddy,2019,Chandur,Chandur,Nalgonda
9,P000010,Bhavani Mudiraj,1512,Patancheru,Patancheru,Sangareddy


In [27]:
tele["name_norm"] = tele["patient_name"].apply(normalize_text)
tele["village_norm"] = tele["village"].apply(normalize_text)
tele["block_norm"] = tele["block"].apply(normalize_text)
tele["district_norm"] = tele["district"].apply(normalize_text)

tele["mobile_last4"] = (
    tele["mobile"]
    .astype("string")
    .str.extract(r"(\d{4})$", expand=False)
)

In [30]:
display(
    tele[
        [
            "tele_source_patient_id",
            "patient_name",
            "mobile_last4",
            "village",
            "block",
            "district"
        ]
    ].head(10)
)


,tele_source_patient_id,patient_name,mobile_last4,village,block,district
0,TEP0000001,Mohan Basha,9751,Chandur,Chandur,Nalgonda
1,TEP0000002,Mohan Laxman,4279,Bodajanampet,Balanagar,Mahabubnagar
2,TEP0000002,Mohan Laxman,<NA>,Bodajanampet,Balanagar,Mahabubnagar
3,TEP0000003,Mohan Narsimha,6710,Kondamallepally,Devarakonda,Nalgonda
4,TEP0000004,Sai Krishna,8147,Macharam,Jadcherla,Mahabubnagar
5,TEP0000005,Anitha Narsimha,2238,Devarakonda,Devarakonda,Nalgonda
6,TEP0000005,Anitha Narsimha,2238,Devarakonda,Devarakonda,Nalgonda
7,TEP0000006,Ramesh Begum,0540,Kangti,Narayankhed,Sangareddy
8,TEP0000006,Ramesh Begum,<NA>,Kangti,Narayankhed,Sangareddy
9,TEP0000007,Suresh Reddy,9019,Chandur,Chandur,Nalgonda


In [32]:
import rapidfuzz as rf
 

In [33]:
from rapidfuzz.fuzz import ratio

def mobile_last4_score(source_value, patient_value):
    if pd.isna(source_value) or pd.isna(patient_value):
        return 0.0

    return 1.0 if str(source_value) == str(patient_value) else 0.0


def text_score(source_value, patient_value):
    if not source_value or not patient_value:
        return 0.0

    return ratio(
        str(source_value),
        str(patient_value)
    ) / 100


def exact_score(source_value, patient_value):
    if not source_value or not patient_value:
        return 0.0

    return 1.0 if source_value == patient_value else 0.0


def age_score(source_age, patient_age):
    if pd.isna(source_age) or pd.isna(patient_age):
        return 0.0

    try:
        difference = abs(
            float(source_age) - float(patient_age)
        )
    except:
        return 0.0

    if difference == 0:
        return 1.0
    elif difference == 1:
        return 0.8
    elif difference == 2:
        return 0.5
    elif difference <= 4:
        return 0.2
    else:
        return 0.0

In [44]:
print("Mobile:")
print(mobile_last4_score("9588", "9588"))
print(mobile_last4_score("9588", "1234"))
print(mobile_last4_score(pd.NA, "9588"))

print("\nText:")
print(text_score("mohan mallalia", "mohan mallalia"))
print(text_score("mohan mallalia", "mohan malalia"))
print(text_score("mohan mallalia", "xyz"))

print("\nExact:")
print(exact_score("chandur", "chandur"))
print(exact_score("chandur", "kandukur"))

print("\nAge:")
print(age_score(40, 40))
print(age_score(40, 41))
print(age_score(40, 42))
print(age_score(40, 50))

Mobile:
1.0
0.0
0.0

Text:
1.0
0.962962962962963
0.0

Exact:
1.0
0.0

Age:
1.0
0.8
0.5
0.0


In [46]:
from collections import defaultdict

village_lookup = defaultdict(set)
block_lookup = defaultdict(set)
district_lookup = defaultdict(set)
mobile_lookup = defaultdict(set)

for idx, row in patients.iterrows():

    if row["village_norm"]:
        village_lookup[row["village_norm"]].add(idx)

    if row["block_norm"]:
        block_lookup[row["block_norm"]].add(idx)

    if row["district_norm"]:
        district_lookup[row["district_norm"]].add(idx)

    if pd.notna(row["mobile_last4"]):
        mobile_lookup[row["mobile_last4"]].add(idx)

In [50]:
def generate_candidates(source_row, patients):

    mobile_last4 = source_row.get("mobile_last4")
    village = source_row.get("village_norm", "")
    block = source_row.get("block_norm", "")
    district = source_row.get("district_norm", "")

    mobile_candidates = set()
    village_candidates = set()
    block_candidates = set()
    district_candidates = set()

    if pd.notna(mobile_last4) and mobile_last4:
        mobile_candidates = mobile_lookup.get(
            str(mobile_last4),
            set()
        )

    if village:
        village_candidates = village_lookup.get(
            village,
            set()
        )

    if block:
        block_candidates = block_lookup.get(
            block,
            set()
        )

    if district:
        district_candidates = district_lookup.get(
            district,
            set()
        )

    candidates = set()

    if mobile_candidates and village_candidates:
        candidates = (
            mobile_candidates &
            village_candidates
        )

    if not candidates and village_candidates and block_candidates:
        candidates = (
            village_candidates &
            block_candidates
        )

    if not candidates and village_candidates:
        candidates = village_candidates

    if not candidates and mobile_candidates:
        candidates = mobile_candidates

    if not candidates:
        candidates = district_candidates

    return patients.loc[list(candidates)].copy()

In [51]:
candidate_counts = []

for _, row in tele.head(100).iterrows():
    candidates = generate_candidates(
        row,
        patients
    )

    candidate_counts.append(len(candidates))

print("Minimum candidates:", min(candidate_counts))
print("Maximum candidates:", max(candidate_counts))
print("Average candidates:", sum(candidate_counts) / len(candidate_counts))
print("Zero-candidate records:", sum(x == 0 for x in candidate_counts))

Minimum candidates: 1
Maximum candidates: 159
Average candidates: 13.75
Zero-candidate records: 0


In [52]:
from rapidfuzz.fuzz import ratio

def mobile_last4_score(source_value, patient_value):
    if pd.isna(source_value) or pd.isna(patient_value):
        return 0.0

    return 1.0 if str(source_value) == str(patient_value) else 0.0


def text_score(source_value, patient_value):
    if not source_value or not patient_value:
        return 0.0

    return ratio(
        str(source_value),
        str(patient_value)
    ) / 100


def exact_score(source_value, patient_value):
    if not source_value or not patient_value:
        return 0.0

    return 1.0 if source_value == patient_value else 0.0


def age_score(source_age, patient_age):
    if pd.isna(source_age) or pd.isna(patient_age):
        return 0.0

    try:
        difference = abs(
            float(source_age) - float(patient_age)
        )
    except (ValueError, TypeError):
        return 0.0

    if difference == 0:
        return 1.0

    if difference == 1:
        return 0.8

    if difference == 2:
        return 0.5

    if difference <= 4:
        return 0.2

    return 0.0

In [55]:
def calculate_linkage_score(source, patient):

    signals = {
        "mobile": (
            mobile_last4_score(
                source["mobile_last4"],
                patient["mobile_last4"]
            ),
            0.20,
            pd.notna(source["mobile_last4"]) and pd.notna(patient["mobile_last4"])
        ),

        "name": (
            text_score(
                source["name_norm"],
                patient["name_norm"]
            ),
            0.35,
            bool(source["name_norm"]) and bool(patient["name_norm"])
        ),

        "village": (
            exact_score(
                source["village_norm"],
                patient["village_norm"]
            ),
            0.15,
            bool(source["village_norm"]) and bool(patient["village_norm"])
        ),

        "block": (
            exact_score(
                source["block_norm"],
                patient["block_norm"]
            ),
            0.10,
            bool(source["block_norm"]) and bool(patient["block_norm"])
        ),

        "district": (
            exact_score(
                source["district_norm"],
                patient["district_norm"]
            ),
            0.05,
            bool(source["district_norm"]) and bool(patient["district_norm"])
        ),

        "gender": (
            exact_score(
                source["gender"],
                patient["gender"]
            ),
            0.10,
            bool(source["gender"]) and bool(patient["gender"])
        ),

        "age": (
            age_score(
                source["age"],
                patient["age_as_of_2026"]
            ),
            0.05,
            pd.notna(source["age"]) and pd.notna(patient["age_as_of_2026"])
        )
    }

    total_score = 0.0
    total_weight = 0.0

    for signal, (score, weight, available) in signals.items():

        if available:
            total_score += score * weight
            total_weight += weight

    if total_weight == 0:
        return 0.0

    return total_score / total_weight